# AI/ML AWS AASTU Cohort — Week 2 Lab: Mini Semantic Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 🎯 Objective
Build a production-grade **Mini Semantic Search Engine** from scratch using:
- **Pandas** for structured FAQ loading and data manipulation
- Custom NLP functions for text cleaning and overlapping text chunking
- **Sentence-Transformers** for generating dense numerical vector embeddings
- **ChromaDB** as the AI-native vector database for indexing and querying
- Retrieve and evaluate the **Top 3 semantic matches** for any natural language inquiry

### Step 0: Install and Import Dependencies
Uncomment and run the cell below if you are running in Google Colab or a fresh virtual environment.

In [ ]:
# %pip install -q pandas numpy chromadb sentence-transformers tabulate

In [ ]:
import os
import re
import json
from typing import List, Dict, Any, Tuple

import pandas as pd
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from tabulate import tabulate

print(f"Pandas version: {pd.__version__}")
print(f"ChromaDB version: {chromadb.__version__}")

## 📂 Step 1: Load FAQ Dataset with Pandas

We load a dataset of curated technical FAQs covering AWS Cloud and AI/ML concepts from CSV or JSON.

In [ ]:
def load_faqs(filepath: str) -> pd.DataFrame:
    """Load FAQ dataset from CSV or JSON into a clean Pandas DataFrame."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Dataset not found at {filepath}")
        
    if filepath.endswith(".csv"):
        df = pd.read_csv(filepath)
    elif filepath.endswith(".json"):
        df = pd.read_json(filepath)
    else:
        raise ValueError("Supported formats are .csv or .json")
        
    return df

# Locate dataset path
dataset_csv_path = os.path.join("data", "faqs.csv")
if not os.path.exists(dataset_csv_path):
    # Fallback if notebook is executed from different working directory
    dataset_csv_path = "faqs.csv"

df_faqs = load_faqs(dataset_csv_path)
print(f"Total FAQs loaded: {len(df_faqs)}")
df_faqs.head(5)

## 🧹 Step 2: Clean and Chunk the Text

Embedding models work best on clean, consistently formatted text.
Furthermore, real-world documents often exceed embedding token limits, so we apply an **overlapping chunking** strategy:
- `chunk_size`: Maximum character length per chunk
- `chunk_overlap`: Overlapping characters between consecutive chunks to avoid cutting off critical context

In [ ]:
def clean_text(text: str) -> str:
    """Normalize whitespaces and remove unwanted characters."""
    if not isinstance(text, str):
        return ""
    # Replace excessive whitespaces and newlines with a single space
    cleaned = re.sub(r"\s+", " ", text)
    return cleaned.strip()


def chunk_text(text: str, chunk_size: int = 180, chunk_overlap: int = 35) -> List[str]:
    """Chunk text using sliding window with overlap to preserve context."""
    cleaned = clean_text(text)
    if not cleaned:
        return []
    if len(cleaned) <= chunk_size:
        return [cleaned]
    
    chunks = []
    start = 0
    step = max(1, chunk_size - chunk_overlap)
    while start < len(cleaned):
        end = min(start + chunk_size, len(cleaned))
        chunk = cleaned[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(cleaned):
            break
        start += step
    return chunks

# Test chunking function on an example
sample_text = df_faqs.iloc[0]["answer"]
sample_chunks = chunk_text(sample_text, chunk_size=120, chunk_overlap=30)
print(f"Original text length: {len(sample_text)} characters")
print(f"Created {len(sample_chunks)} chunks:")
for i, ch in enumerate(sample_chunks):
    print(f"  [Chunk {i+1}]: {ch}")

### Prepare Documents and Metadata for Vector Database

In [ ]:
documents: List[str] = []
metadatas: List[Dict[str, Any]] = []
ids: List[str] = []

for _, row in df_faqs.iterrows():
    faq_id = int(row["id"])
    question = clean_text(str(row["question"]))
    answer = clean_text(str(row["answer"]))
    category = clean_text(str(row.get("category", "General")))
    
    # Combine question and answer for holistic semantic representation
    combined_text = f"Question: {question} Answer: {answer}"
    chunks = chunk_text(combined_text, chunk_size=200, chunk_overlap=40)
    
    for chunk_idx, chunk in enumerate(chunks):
        doc_id = f"faq_{faq_id}_chunk_{chunk_idx}"
        documents.append(chunk)
        metadatas.append({
            "faq_id": faq_id,
            "question": question,
            "category": category,
            "chunk_index": chunk_idx,
            "total_chunks": len(chunks)
        })
        ids.append(doc_id)

print(f"Total documents (chunks) prepared for indexing: {len(documents)}")
print(f"Sample Document ID: {ids[0]}")
print(f"Sample Document Text: {documents[0]}")
print(f"Sample Metadata: {metadatas[0]}")

## 🧠 Step 3: Embed and Store in ChromaDB

We initialize **ChromaDB** with the pre-trained `all-MiniLM-L6-v2` embedding function from Sentence Transformers and cosine similarity metric.

In [ ]:
# Initialize ChromaDB Persistent Client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Setup Sentence Transformer Embedding Function
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Get or create collection with cosine distance metric
collection = chroma_client.get_or_create_collection(
    name="aws_week_2_faqs",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# Index documents into ChromaDB
collection.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully indexed {collection.count()} chunks into collection '{collection.name}'!")

## 🔍 Step 4: Query Semantic Search & Print Top 3 Matches

We query the collection using natural language questions and retrieve the **Top 3** most relevant matches.

In [ ]:
def search_faqs(query: str, top_k: int = 3) -> None:
    """Query Chroma collection and print nicely formatted top 3 matches."""
    cleaned_query = clean_text(query)
    results = collection.query(
        query_texts=[cleaned_query],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    print("=" * 85)
    print(f"🔎 SEARCH QUERY: \"{cleaned_query}\"")
    print("=" * 85)
    
    table_data = []
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]
    
    for rank, (doc, meta, dist) in enumerate(zip(docs, metas, distances), start=1):
        # Cosine similarity = 1 - cosine distance
        similarity = max(0.0, 1.0 - dist)
        table_data.append([
            f"#{rank}",
            f"{similarity:.4f}",
            meta.get("category", "General"),
            meta.get("question", ""),
            (doc[:90] + "...") if len(doc) > 90 else doc
        ])
        
    headers = ["Rank", "Sim Score", "Category", "FAQ Question", "Matched Excerpt"]
    print(tabulate(table_data, headers=headers, tablefmt="grid"))
    print("\n")

### Query 1: Cloud ML Model Deployment
Notice how it finds **Amazon SageMaker** even when the query uses synonyms.

In [ ]:
search_faqs("How can I deploy and train machine learning models on AWS?", top_k=3)

### Query 2: Serverless Execution Limits
Retrieves **AWS Lambda** based on operational constraints.

In [ ]:
search_faqs("What serverless tool has a 15-minute execution limit?", top_k=3)

### Query 3: Vector Embeddings and Conceptual Search
Demonstrates conceptual retrieval regarding vector stores and embeddings.

In [ ]:
search_faqs("Why is semantic search better than keywords for synonyms and intent?", top_k=3)

### Query 4: Hardware and GPU Acceleration
Retrieves **EC2 GPU instance families** (G-series and P-series).

In [ ]:
search_faqs("Which GPU instances are best for distributed deep learning training?", top_k=3)

## 📊 Step 5: Summary & Key Takeaways

| Feature | Traditional Keyword Search | Mini Semantic Search (ChromaDB) |
| :--- | :--- | :--- |
| **Matching Mechanism** | Exact words / n-grams (lexical) | Dense vector geometric proximity (semantic) |
| **Synonyms & Paraphrasing**| ❌ Fails without manual thesaurus | ✅ Automatically understood through embeddings |
| **Context Preservation** | ❌ None | ✅ Maintained with chunking + chunk overlap |
| **Storage Engine** | Inverted indices | HNSW vector index in ChromaDB |

✅ **Lab Deliverable Complete!**